# Reproducibility Notebook: Implicit Event Prioritization in Large Language Models

This notebook reproduces the experiments and analyses reported in the study of implicit event prioritization in large language models (LLMs) using real Formula 1 timing streams.
It covers the **main experiment** and an additional **controlled timing-rank follow-up** used to clarify the numerical-direction result.

## 1. Overview

**Research question.** When an LLM is shown a short stream of structured events and is asked which three are most important to report right now,
without any rule about what is important, which events does it select, and what does that selection depend on?

**Main experiment.** Real Formula 1 timing windows are turned into presentations in which one focal event is placed at controlled positions. Three conditions isolate
event type, numerical direction and candidate-set size; serial position is manipulated in all of them. Three Qwen3 models (4B, 8B, 14B) answer one fixed prompt.
Selection of the focal event is compared with a uniform random-choice baseline. No human saliency labels are assumed.

**Controlled timing-rank follow-up.** The main experiment showed that faster focal timing events were selected much more often than slower ones. Because the focal numerical
outlier was also necessarily an extreme value of its list, a controlled follow-up uses the same real source windows and changes the ordinal timing status of the same focal event
(fastest or slowest in its comparison set) while serial position is varied independently between first and last.

**Workflow of this notebook**

| Section | Content | Needs |
|---|---|---|
| 2 | Environment and configuration: choose the LLM and the options | nothing |
| 3 | Data and event-stream construction: verify or rebuild the input data | rebuild: FastF1 download |
| 4 | Main experiment design | nothing |
| 5 | Main experiment inference (or loading of existing responses) | the LLM server, only if you run inference |
| 6 | Main experiment results | nothing |
| 7 | Controlled timing-rank follow-up: design | nothing |
| 8 | Controlled follow-up results | nothing |
| 9 | Integrated findings | nothing |
| 10 | Reproducibility outputs | nothing |

The pipeline runs for **one LLM at a time**. Run it again with another LLM configuration to add another LLM; the analysis combines every run it finds in `results/`
and can add the three shipped reference runs for comparison. Without a running LLM the notebook works on the shipped reference responses (`RUN_INFERENCE = False`).

## 2. Environment and configuration

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "run.py").exists():
    if ROOT.parent == ROOT:
        raise SystemExit("open the notebook inside the release folder")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
TAG = "notebook"


def opt(name, default):
    v = os.environ.get("NB_" + name)
    if v is None:
        return default
    if isinstance(default, bool):
        return v.lower() in ("1", "true", "yes")
    return type(default)(v)


# The two experiment parts: the main experiment and the controlled timing-rank follow-up.
MAIN, FOLLOWUP = "main_experiment", "controlled_timing_rank_followup"

# ---------------- the choices (edit here; environment variables NB_<NAME> override them for headless runs) ----------------
LLM_CONFIG = opt("LLM_CONFIG", "configs/llm_qwen3-8b.yaml")   # one LLM per run: a configs/llm_*.yaml file or your own copy of a template
RUN_INFERENCE = opt("RUN_INFERENCE", False)                    # True: really call the LLM (hours for a full run; resumable)
PARTS_TO_RUN = [MAIN, FOLLOWUP]                                      # main experiment (1,040 presentations) and controlled follow-up (252 presentations)
LIMIT = opt("LIMIT", 0) or None                                # e.g. 20 for a smoke test (kept apart as <name>__smoke and not analysed)
INPUT_MODE = opt("INPUT_MODE", "shipped")                      # shipped | rebuild_from_primitive | rebuild_download
COMBINE_WITH_REFERENCE = opt("COMBINE_WITH_REFERENCE", True)   # add the shipped Qwen3 4B/8B/14B runs to the tables and figures
RESULTS_DIR = Path(opt("RESULTS_DIR", "results"))
OUT_DIR = Path(opt("OUT_DIR", "outputs"))                       # flat folder: every file is <TAG>__<name>.<ext>
INCLUDE_MOCK = opt("INCLUDE_MOCK", False)                      # only for pipeline tests with the offline mock backend
# --------------------------------------------------------------------------------------------------------------------------

import pandas as pd
from IPython.display import Image, Markdown, display
from llm_config import load_llm_config
from paths import INPUTS, REFERENCE

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 60)
cfg = load_llm_config(LLM_CONFIG)
print(f"LLM: {cfg.name}  backend: {cfg.backend}  model: {cfg.model}")
print("inference options:", cfg.options, " thinking:", cfg.think)
print("deviations from the frozen design:", cfg.deviations_from_frozen or "none")

The reference runs use Qwen3 4B (Q4_K_M), 8B and 14B through Ollama with one fixed prompt and the frozen inference settings: temperature 0, seed 42, context 4,096,
at most 48 output tokens, thinking off. Seeds: dataset sampling 42, presentation order 20250927, cluster bootstrap 7 (2,000 resamples), follow-up bootstrap 31 and 5 (B = 5,000).
Any deviation you choose for another LLM is printed above and stored with its responses.

## 3. Data and event-stream construction

The input data are the 41,947 primitive events (lap completions, sector crossings, pit in and out) of 12 races of the 2025 season, 10,627 candidate windows of the trailing 10 s,
and 90 seed windows (45 for event type, 45 for numerical value). Drivers are anonymised and event ids are opaque three-letter labels.
The inputs are shipped in `inputs/` with SHA-256 hashes in `inputs/MANIFEST.sha256`. The module `si_*` (`scripts/make_inputs.py`) rebuilds them from raw timing data
(`rebuild_download`, needs FastF1 and internet) or from the shipped primitive events (`rebuild_from_primitive`, offline); rebuilt files are compared byte by byte with the shipped ones.

In [ ]:
def sh(script, *args):
    r = subprocess.run([sys.executable, str(ROOT / "scripts" / script), *args], capture_output=True, text=True, cwd=ROOT)
    print(r.stdout[-3000:] + r.stderr[-1500:])
    return r.returncode


print("== structural verification of the shipped inputs ==")
assert sh("verify_inputs.py", "--inputs", str(INPUTS)) == 0
if INPUT_MODE != "shipped":
    rebuilt = ROOT / "inputs_rebuilt"
    src = ["--download", "--cache", str(ROOT / ".fastf1_cache")] if INPUT_MODE == "rebuild_download" else ["--from-primitive", str(INPUTS)]
    print("== rebuild the inputs ==")
    sh("make_inputs.py", "--out", str(rebuilt), *src, "--compare", str(INPUTS))
    print("The rebuilt inputs are in", rebuilt, "- they are not used below unless you point INPUTS at them.")

In [ ]:
import an_design as design
import an_figures as F
import an_style as style
style.apply_theme()
style.PREFIX = TAG + "__"
FIG = TAB = OUT_DIR
display(Markdown("**Figure 1.** Study framework for implicit LLM-based event prioritization."))
display(Image(str(OUT_DIR / "reference__study_framework_board.png"), width=640))
display(Markdown("**Table 1.** Dataset and experimental composition."))
display(design.dataset_table(INPUTS))

## 4. Main experiment design

Each presentation is a list of 5, 10 or 20 events from one 10 s window (or a window-derived list), written as one JSON object per line. One event is the **focal event**; it is never
marked in the prompt. The LLM sees only the list and one fixed instruction, and answers with the three ids it considers most important.

### 4.1 Event-type condition
In 45 seed windows, one event type differs from the rest of the window (for example a pit event among lap events). The focal event is the minority-type event. 225 presentations.

### 4.2 Numerical-direction condition
In 45 seed windows, one timing value is a robust outlier within its comparison class, either faster or slower than the others. The focal event is the outlier; the comparison list contains
only values of the same class. A faster focal value is the lowest value of its list and a slower focal value is the highest, so direction and extremeness coincide in this condition. 225 presentations.

### 4.3 Candidate-set-size condition
The event-type focal event is shown among 5, 10 or 20 events. 590 presentations.

### 4.4 Serial-position manipulation
In every condition the focal event is placed at five positions (first, 1/4, 1/2, 3/4, last), with the order of the other events fixed by the presentation seed.

### 4.5 Response task and parsing
The answer must be exactly three distinct displayed ids. One frozen parser (`parsing.py`) reads every answer; anything else is an invalid response and is reported, never repaired.
The baseline is uniform random choice (3/n for Top-3, 1/n for Top-1). Rates use format-valid responses; confidence intervals are 95% percentile cluster bootstraps over seed windows.

In [ ]:
import json
from an_figures import read_jsonl, EXPERIMENT_PARTS
from conditions import EVENT_TYPE, condition_of

trial = next(t for t in read_jsonl(INPUTS / EXPERIMENT_PARTS["main_experiment"][0])
             if condition_of(t) == EVENT_TYPE and t["n_events"] in (7, 8) and abs(t["focal_position_frac"] - 0.5) < 1e-9
             and t["focal_type"] == "PitIn" and t["types_in_order"].count("LapComplete") == t["n_events"] - 1)
abbr = {"LapComplete": "LAP", "SectorCross": "SEC", "PitIn": "PIT-IN", "PitOut": "PIT-OUT"}
rows = []
for i, e in enumerate((json.loads(l.strip()) for l in trial["prompt"].split("\n") if l.strip().startswith("{") and '"id"' in l), 1):
    tv = e.get("lap_time_s", e.get("sector_time_s"))
    rows.append({"No.": i, "ID": e["id"], "Type": abbr[e["type"]], "Driver": e["driver"], "Lap": e.get("lap", "–"),
                 "Time (s)": f"{tv:.3f}" if tv is not None else "–"})
display(Markdown("**Example.** Presentation from the event-type condition in human-readable form. The model receives the same information as JSON lines; the focal event is not marked in the prompt."))
display(pd.DataFrame(rows).style.hide(axis="index"))


## 5. Main experiment inference

`run_llm` sends each presentation to the LLM with the frozen settings and appends every answer to `results/<name>__*.jsonl`. It can be interrupted and started again
(finished trials are skipped). A full run is 1,292 requests (1,040 main experiment, 252 follow-up). Set `LIMIT` for a smoke test first.
Nothing is sent anywhere unless `RUN_INFERENCE = True`.

In [ ]:
from run import run_llm
if RUN_INFERENCE:
    run_dir = run_llm(cfg, parts=tuple(PARTS_TO_RUN), inputs_dir=INPUTS, results_root=RESULTS_DIR, limit=LIMIT)
    print("results folder:", run_dir)
else:
    print("RUN_INFERENCE is False: no LLM is called. Existing runs in", RESULTS_DIR, "and the shipped reference runs are analysed.")

In [ ]:
from cli import select_runs
import an_integrity as integrity
runs = select_runs(RESULTS_DIR, COMBINE_WITH_REFERENCE, False, None, include_smoke=False, include_mock=INCLUDE_MOCK)
# has_main_experiment / has_followup are frozen identifiers; public terminology: main experiment / controlled timing-rank follow-up
display(pd.DataFrame([{"run": r.name, "label": r.label, "main experiment": r.has_main_experiment, "controlled timing-rank follow-up": r.has_followup, "reference run": r.reference} for r in runs]))
checks = integrity.check_runs(runs, INPUTS)
display(checks[~checks.ok] if not checks.ok.all() else Markdown(f"All {len(checks)} integrity checks passed."))
assert checks.ok.all(), "fix the failed checks before interpreting results (an incomplete run fails 'run is complete')"

### Compute every table and figure
`build_report` computes everything from the response files in one pass (cluster bootstrap, GEE, paired bootstrap, design diagnostics) and writes figures
(PNG, PDF, EPS, grayscale proofs) and tables (CSV) to `outputs/`, every file named `notebook__<name>.<ext>`. It takes about a minute.

In [ ]:
from an_report import build_report
summary = build_report(runs, OUT_DIR, tag=TAG, inputs=INPUTS, log=lambda *_: None)
print({k: v for k, v in summary.items() if k in ("runs", "integrity_all_ok", "seconds")})


def tab(name):
    return pd.read_csv(TAB / f"{TAG}__{name}.csv")


def fig(name, w=560, caption=""):
    if caption:
        display(Markdown(caption))
    display(Image(str(FIG / f"{TAG}__{name}.png"), width=w))

## 6. Main experiment results

### 6.1 Response integrity

In [ ]:
display(Markdown("Integrity checks of every response file against the inputs (all must be True) and the independent re-parse of every raw answer."))
display(checks.groupby(["run", "experiment_part"]).ok.agg(["sum", "count"]).rename(columns={"sum": "passed", "count": "checks"}))
display(tab("parser_audit"))

### 6.2 Numerical direction
Faster focal values are ranked first much more often than slower ones, in every model, while pooled Top-3 selection over both directions stays near chance.

In [ ]:
display(Markdown("**Table 2.** Focal selection by condition (Top-1, Top-3, lift over the uniform random baseline) and, for the numerical condition, by direction."))
display(tab("focal_selection_by_condition_and_model"))
display(tab("timing_direction_by_model"))
fig("timing_direction_top1_selection", caption="**Figure 3.** Numerical direction: faster focal values are ranked first, slower ones are not (chance is about 16%).")

### 6.3 Serial position
Selection of the focal event depends on where it is placed. The curves differ between conditions and between models, so the effect is described as a serial-position effect (a last-position advantage that varies by condition and model).

In [ ]:
fig("serial_position_top3_inclusion", 700, "**Figure 4.** Serial-position response curves: Top-3 selection by focal position in the three conditions.")
display(Markdown("**Table 3.** Pooled logistic GEE (exchangeable working correlation, clusters = seed windows). Positions are compared with the middle positions; the first listed LLM is the reference."))
display(tab("pooled_gee_effects"))
fig("pooled_gee_position_effects_forest", caption="**Figure 5.** First and last position versus middle positions (pooled odds ratios, log scale).")

### 6.4 Candidate-set size

In [ ]:
fig("candidate_set_size_top3_inclusion", caption="**Figure 6.** Candidate-set size: Top-3 selection with 5, 10 and 20 events.")
fig("candidate_set_size_chance_adjusted_lift", caption="**Figure 7.** Lift over chance by candidate-set size.")

### 6.5 Event type

In [ ]:
fig("event_type_pit_event_selection", caption="**Figure 8.** Event type: pit events as the focal event.")
display(tab("event_type_pit_events"))

### 6.6 Model heterogeneity
The pooled effects coexist with differences between models. The Position x LLM interaction is tested and the position effects are also fitted separately per LLM.

In [ ]:
display(Markdown("**Table 4.** Position effects fitted separately per LLM, and the Position x LLM interaction test."))
display(tab("serial_position_effects_by_model"))
display(tab("serial_position_by_model_interaction"))
fig("serial_position_effects_by_model", 700, "**Supplementary Figure S1.** Position odds ratios per LLM.")

### 6.7 Response validity and Top-1 agreement

In [ ]:
fig("response_format_quality_by_model", 700, "**Figure 9.** Main experiment output quality: invalid answers, invented identifiers, top-1 agreement across positions.")
display(tab("response_quality_and_top1_agreement"))
display(tab("top1_agreement_by_condition_and_model"))

### 6.8 Statistical robustness
The conclusions do not depend on the estimator, on the denominator, or on the choice of seeds: naive logit, cluster-robust logit and GEE are compared, valid-only and all-trial rates are shown side by side,
and the candidate-set-size condition is repeated on the seeds available at every list size. The design diagnostics document what the conditions can and cannot separate.

In [ ]:
for n, t in [("estimator_comparison_wide", "Supplementary Table S1. Naive logit, cluster-robust logit and GEE"), ("valid_only_versus_all_trial_rates", "Supplementary Table S2. Valid-only versus all-trial rates"),
             ("candidate_set_size_complete_seeds", "Supplementary Table S3. Candidate-set size on seeds available at every list size"), ("candidate_set_size_complete_seeds_log_odds", "Supplementary Table S4. Log list-size odds ratio on complete seeds"),
             ("lowest_displayed_value_preference", "Supplementary Table S5. Preference for the lowest displayed timing value"), ("design_direction_balance", "Supplementary Table S6. Balance of faster and slower seeds"),
             ("design_event_representation", "Supplementary Table S7. How each event type is written in the prompt"), ("design_candidate_set_size_composition", "Supplementary Table S8. Composition of the background by list size")]:
    display(Markdown(f"**{t}**"))
    display(tab(n))
for n, t in [("candidate_set_size_lift_complete_seeds", "Supplementary Figure S2. Lift on complete seeds"), ("lowest_displayed_value_preference", "Supplementary Figure S3. Lowest-value preference"),
             ("estimator_comparison", "Supplementary Figure S4. Estimator comparison"), ("design_balance", "Supplementary Figure S5. Design balance")]:
    fig(n, 600, f"**{t}**")

## 7. Controlled timing-rank follow-up

The main experiment showed a strong association between numerical direction and focal-event selection. Because the focal numerical outlier was also necessarily an extreme value within its list,
an additional controlled experiment tests the effect of ordinal timing status more directly. To distinguish the association from a generic effect of numerical extremeness, the follow-up uses the
same real source windows.

**Design.** For each eligible source window, the focal timing value is modified so that the same event becomes either the fastest or the slowest item in its comparison set, while all other event
content is held fixed:

* faster = min(original, min of the other values) - eps, slower = max(original, max of the other values) + eps, with eps = 1.0 x the robust scale (1.4826 x MAD, floor 50 ms) of the other values;
* the modified value must lie within the range observed for that race and comparison class; 3 of the 45 numerical source windows fail this rule and are excluded, leaving **42 source windows**;
* each source window is shown six times: focal timing (original, fastest, slowest) x serial position (first, last), giving **252 presentations per model** and 756 responses for the three reference models;
* the same real source windows, event schema, field semantics, prompt, models and inference settings are used as in the main experiment.

This design permits a within-window comparison of numerical rank and presentation position. It is an additional controlled follow-up, not an independent second study.

In [ ]:
display(Markdown("**Table 9.** Source windows excluded by the plausibility rule."))
display(tab("controlled_timing_rank_followup_excluded_sources"))

## 8. Controlled follow-up results

Effects are percentage points of Top-3 (or Top-1) probability with a 95% paired bootstrap over source windows (B = 5,000). The pooled model is a logistic GEE,
`top3 ~ content * position + model`, clustered by source window.

### 8.1 Integrity and response quality

In [ ]:
fu = checks[checks.experiment_part == "controlled timing-rank follow-up"]
display(Markdown(f"Follow-up integrity checks passed: {int(fu.ok.sum())} of {len(fu)}."))
display(Markdown("**Table 8.** Follow-up response quality (share of responses with valid format, with invented ids, and naming a driver instead of an event id)."))
display(tab("controlled_timing_rank_followup_response_quality"))

### 8.2 Descriptive Top-3 and Top-1

In [ ]:
display(Markdown("**Table 5.** Follow-up Top-1 and Top-3 (%) by focal timing and position, with 95% paired bootstrap intervals."))
display(tab("controlled_timing_rank_followup_descriptive"))
fig("controlled_timing_rank_followup_top3_inclusion", 700, "**Figure 10.** Controlled timing-rank follow-up: Top-3 by focal timing and position.")
fig("controlled_timing_rank_followup_top1_selection", 700, "**Figure 11.** Controlled timing-rank follow-up: Top-1 by focal timing and position.")

### 8.3 Paired bootstrap and pooled GEE

In [ ]:
eff = tab("controlled_timing_rank_followup_marginal_effects_pp")
display(Markdown("**Table 6.** Marginal effects in percentage points (paired bootstrap, 95% CI)."))
mask = (eff.effect.str.endswith("| mean") | eff.effect.str.contains("interaction")) & eff.group.isin([r.name for r in runs])
display(eff[mask])
fig("timing_rank_vs_serial_position_effects", caption="**Figure 12.** Numerical-rank effect (fastest − slowest) and serial-position effect (last − first) on Top-3 selection, by model (percentage points, 95% paired-bootstrap CI). Filled markers and solid bars: numerical rank; open markers and dashed bars: serial position.")
display(Markdown("**Table 7.** Pooled GEE `top3 ~ content * position + model` (odds ratios; a model with separation is flagged and must not be interpreted)."))
display(tab("controlled_timing_rank_followup_gee"))

### 8.4 Content versus presentation, and model-size pattern
Each panel shows the same source windows: the vertical gap between lines is the effect of the focal timing, the slope of each line is the effect of moving the same event from first to last.

In [ ]:
fig("timing_rank_vs_serial_position_crossover", 700, "**Figure 13.** Content versus presentation effects by model.")

Check of the computed follow-up values against the reported values (descriptive Top-3 / Top-1, paired-bootstrap effects, pooled GEE):

In [ ]:
import numpy as np
REFERENCE_TRIO = sorted(r.name for r in runs if r.reference) == ["qwen3-14b", "qwen3-4b-q4_K_M", "qwen3-8b"]
if REFERENCE_TRIO:
    desc = tab("controlled_timing_rank_followup_descriptive")
    exp_desc = {  # (content, position): (Top-3, Top-1) in %, as reported
        "qwen3-4b-q4_K_M": {("original", "first"): (29, 12), ("original", "last"): (90, 69), ("faster", "first"): (29, 12), ("faster", "last"): (95, 86), ("slower", "first"): (12, 0), ("slower", "last"): (62, 36)},
        "qwen3-8b": {("original", "first"): (30, 28), ("original", "last"): (82, 70), ("faster", "first"): (55, 48), ("faster", "last"): (98, 88), ("slower", "first"): (2, 0), ("slower", "last"): (46, 32)},
        "qwen3-14b": {("original", "first"): (43, 38), ("original", "last"): (60, 52), ("faster", "first"): (76, 62), ("faster", "last"): (88, 82), ("slower", "first"): (2, 0), ("slower", "last"): (17, 10)}}
    bad = []
    for m, cells in exp_desc.items():
        for (c, p), (t3, t1) in cells.items():
            r = desc[(desc.llm == m) & (desc.content == c) & (desc.position == p)].iloc[0]
            if (round(r.top3_pct), round(r.top1_pct)) != (t3, t1):
                bad.append((m, c, p, round(r.top3_pct), round(r.top1_pct), t3, t1))
    exp_eff = {"qwen3-4b-q4_K_M": (25.2, 15.8, 34.5, 59.2, 48.0, 70.3), "qwen3-8b": (51.9, 42.1, 61.8, 46.3, 34.8, 57.5), "qwen3-14b": (72.1, 60.3, 82.9, 14.3, 5.7, 23.8)}
    for m, v in exp_eff.items():
        a = eff[(eff.group == m) & (eff.outcome == "top3") & (eff.effect == "faster-slower | mean")].iloc[0]
        b = eff[(eff.group == m) & (eff.outcome == "top3") & (eff.effect == "last-first | mean")].iloc[0]
        got = tuple(round(x, 1) for x in (a.pp, a.ci_lo_pp, a.ci_hi_pp, b.pp, b.ci_lo_pp, b.ci_hi_pp))
        if got != v:
            bad.append((m, "effects", got, v))
    gee = tab("controlled_timing_rank_followup_gee")
    g = gee[gee.spec == "additive_llm"].set_index("term")
    for term, v in {"content[faster]": (2.28, 1.43, 3.63), "content[slower]": (0.12, 0.05, 0.30), "position[last]": (6.81, 4.51, 10.28)}.items():
        got = tuple(round(g.loc[term, c], 2) for c in ("OR", "ci_lo", "ci_hi"))
        if got != v:
            bad.append((term, got, v))
    print("values equal the reported ones:", not bad, bad or "")
    print("GEE observations:", int(g.n.iloc[0]), " clusters:", int(g.clusters.iloc[0]))
else:
    print("The comparison with the reported values is made only when the three reference runs are analysed (COMBINE_WITH_REFERENCE = True).")

## 9. Integrated findings

The main experiment establishes that implicit event saliency is associated with event type, numerical direction, candidate-set size, serial position and model configuration.
The controlled timing-rank follow-up strengthens the interpretation of two effects for the tested prompt and model configurations.

**Numerical content.** Making the same focal timing record the fastest rather than the slowest item in the same comparison set increases its Top-3 selection probability, by roughly
25 pp for 4B, 52 pp for 8B and 72 pp for 14B.

**Presentation order.** Moving the same event from the first to the last position increases its Top-3 selection probability, by roughly 59 pp for 4B, 46 pp for 8B and 14 pp for 14B.

**Model-specific pattern.** For 4B the presentation-position effect exceeds the numerical-rank effect, for 8B the two are comparable, and for 14B the numerical-rank effect is much larger than the
position effect. The relative influence of content and presentation shifts markedly across model sizes. This is a description of the observed selections, not an inference about internal mechanisms
or about understanding, and it holds for the tested prompt, models and inference settings only. Content x position interactions and the model main effect are not statistically significant in the pooled GEE.

In [ ]:
eff = tab("controlled_timing_rank_followup_marginal_effects_pp")
rows = []
for r in runs:
    if not r.has_followup:
        continue
    d = eff[(eff.group == r.name) & (eff.outcome == "top3")].set_index("effect")
    c, p = d.loc["faster-slower | mean"], d.loc["last-first | mean"]
    rows.append({"LLM": r.label, "numerical rank: fastest - slowest (pp)": round(c.pp, 1), "95% CI": f"[{c.ci_lo_pp:.1f}, {c.ci_hi_pp:.1f}]",
                 "serial position: last - first (pp)": round(p.pp, 1), "95% CI ": f"[{p.ci_lo_pp:.1f}, {p.ci_hi_pp:.1f}]"})
display(Markdown("**Table 10.** Content and presentation effects on Top-3 by model."))
display(pd.DataFrame(rows))

## 10. Reproducibility outputs

Everything is already on disk in `outputs/`: figures as PNG, PDF, EPS and grayscale proofs (`*_gray.png`), and tables as CSV, all named `notebook__<name>.<ext>`.
`outputs/notebook__report.txt` lists every file and `outputs/notebook__REPORT_MANIFEST.json` holds the headline numbers. The pre-generated analysis of the reference responses is `outputs/reference__*`.
To compare your LLM with the reference, keep `COMBINE_WITH_REFERENCE = True`: your LLM appears as an additional line in every figure and row in every table.

A separate audit of byte-identical original-condition prompts found identical Top-1 and focal Top-3 outcomes in all 84 paired trials per model; the rerun response files are not included in this release, so this result is not recomputed here.

In [ ]:
import hashlib
display(Markdown("**SHA-256 of the frozen configuration files**"))
display(tab("frozen_config_hashes"))
display(Markdown("**SHA-256 of the response files that were analysed**"))
rows = []
for r in runs:
    for ph in PARTS_TO_RUN:
        f = r.file(ph)
        if f.exists():
            rows.append({"run": r.name, "part": "main experiment" if ph == MAIN else "controlled timing-rank follow-up", "file": f.name, "sha256": hashlib.sha256(f.read_bytes()).hexdigest()})
display(pd.DataFrame(rows))
display(Markdown(f"Inputs match `inputs/MANIFEST.sha256`: **{bool(checks[checks.run == 'inputs'].ok.all())}**"))
print((OUT_DIR / f"{TAG}__report.txt").read_text(encoding="utf-8")[:1200])